# Aula 09 · Do problema ao sistema

Esta aula apresenta o [capítulo 9 do site](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/09-do-problema-ao-sistema/). A ideia central: **montar um sistema é traduzir** — cada incógnita vira uma coluna, cada informação vira uma linha — e a resposta só vale depois de conferida no problema, não no sistema.

**Ao fim da aula você consegue:**

1. nomear as incógnitas e escrever uma equação por informação;
2. arrumar as equações numa matriz e resolver com o `np.linalg.solve` dos capítulos anteriores;
3. montar sistemas de comparações, de conservação, de circuitos e de forças;
4. reconhecer quando as informações não bastam ou se contradizem.

**Roteiro:** 🧩 · 1. a receita · 2. 🧑‍🏫 tabela e matriz · 3. comparações · 4. conservação · 5. circuitos · 6. forças · 7. informação que falta · 8. outra área · 🎯 prática · 🧩 o quarteirão · 📋 a lista · 🚪

O caderno ocupa mais de um encontro: pare onde a aula terminar e continue daí na
seguinte.

## Como usar este caderno

- **Rode a célula ⚙️** logo abaixo antes de tudo (e de novo se o Colab reiniciar).
- **🧑‍🏫 No quadro:** a dedução é feita à mão, no quadro. Acompanhe **no seu
  caderno de papel** — é o mesmo tipo de conta que cai na parte em papel da prova.
  O resumo fica recolhido aqui, para conferir depois.
- **🧰 Comando novo:** antes do primeiro uso de um comando de `numpy` ou
  `matplotlib`, uma caixa explica o que ele faz. Rode a célula de exemplo logo
  abaixo dela.
- **✍️ Passo:** o código é escrito ao vivo, em pedaços pequenos — a instrução
  está logo acima de cada célula vazia. Estudando sozinho, escreva você mesmo; o
  código completo está no capítulo do site (links 📖).
- Depois de escrever e **antes de rodar**, registre a sua previsão. Só então rode
  e abra o **▶ O que aconteceu**. A previsão errada é a parte que ensina — não a
  apague.
- **🎯 Sua vez:** escreva a função no lugar de `# sua solução aqui` e rode a
  célula `confere` logo abaixo: ✅ acertou, ❌ ainda não. Tente antes de abrir a
  💡 Dica.
- O caderno pode ocupar mais de uma aula: continue de onde parou, rodando antes a
  célula ⚙️ e as células 📦.

In [ ]:
# ⚙️ Rode esta célula antes de tudo. Ela prepara a correção automática dos
# exercícios 🎯 — não precisa ler (usa coisas que não fazem parte do curso).
import math


def _mostra(argumentos):
    textos = []
    for a in argumentos:
        textos.append(a.__name__ if callable(a) else repr(a))
    return ", ".join(textos)


def _numero(x):
    try:
        float(x)
        return not isinstance(x, (str, bool))
    except (TypeError, ValueError):
        return False


def _igual(veio, esperado, tol):
    # Número: compara com tolerância relativa, porque conta com float quase
    # nunca bate na última casa. Lista, tupla ou array: item a item.
    if _numero(esperado) and _numero(veio):
        return math.isclose(float(veio), float(esperado), rel_tol=tol, abs_tol=tol)
    if isinstance(esperado, (list, tuple)) and hasattr(veio, "__len__") and not isinstance(veio, str):
        if len(veio) != len(esperado):
            return False
        return all(_igual(v, e, tol) for v, e in zip(veio, esperado))
    return veio == esperado


def confere(funcao, casos, tol=1e-6):
    """Chama funcao com cada caso (argumentos, esperado) e diz se acertou."""
    certos = 0
    for numero, (argumentos, esperado) in enumerate(casos, start=1):
        chamada = f"{funcao.__name__}({_mostra(argumentos)})"
        try:
            veio = funcao(*argumentos)
        except Exception as erro:
            print(f"❌ {chamada} deu erro: {type(erro).__name__}: {erro}")
            continue
        if veio is not None and _igual(veio, esperado, tol):
            certos += 1
            print(f"✅ {chamada} devolveu {veio!r}")
        elif veio is None:
            print(f"❌ {chamada} devolveu None — faltou o return?")
        else:
            print(f"❌ {chamada} devolveu {veio!r}, mas devia ser {esperado!r}")
    print(f"{certos} de {len(casos)} certos")


def confere_valor(nome, valor, esperado, tol=1e-6):
    """Diz se a variável `nome` ficou com o valor esperado."""
    if valor is not None and _igual(valor, esperado, tol):
        print(f"✅ {nome} = {valor!r}")
    else:
        print(f"❌ {nome} vale {valor!r}, mas devia ser {esperado!r}")

# --- bibliotecas desta aula ---
import numpy as np
import matplotlib.pyplot as plt

## 🧩 O problema da aula

> **Trânsito — as ruas sem sensor.**
>
> *Um quarteirão do centro é contornado por quatro ruas de mão única, que formam um
> anel: de A para B ($x_1$), de B para C ($x_2$), de C para D ($x_3$) e de D para A
> ($x_4$). Em cada cruzamento, carros também entram e saem do anel por outras ruas,
> e essas foram contadas (carros por hora):*
>
> | cruzamento | entram de fora | saem para fora |
> |---|---|---|
> | A | 400 | 150 |
> | B | 300 | 250 |
> | C | 200 | 350 |
> | D | 100 | 250 |
>
> *A prefeitura só tem **um** sensor no anel, na rua de D para A: 180 carros por hora.
> "**Quantos carros passam nas outras três ruas?** Precisamos saber onde pôr o
> semáforo novo."*

Em cada cruzamento, os carros que chegam são os que saem. No fim da aula, você monta
esse sistema e descobre por que os quatro cruzamentos, sozinhos, **não** bastam.

## 1. Uma receita em quatro passos

Três compras na feira, e a nota só mostra o total. Compra 1: 2 kg de maçã, 1 de
banana e 3 de laranja, R\$ 14,00. Compra 2: 1, 3 e 2 kg, R\$ 13,50. Compra 3: 3, 2 e
1 kg, R\$ 11,50. Quanto custa o kg de cada fruta?

1. **nomeie** as incógnitas, com unidade: $x_0, x_1, x_2$ = R\$/kg de maçã, banana e
   laranja;
2. **uma equação por informação**: cada compra é uma soma igual ao total;
3. **arrume**: uma coluna por incógnita, na mesma ordem, zeros explícitos,
   constantes à direita;
4. **resolva e confira no problema**.

📖 [capítulo 9 · Uma receita em quatro passos](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/09-do-problema-ao-sistema/#uma-receita-em-quatro-passos)

Lembretes da aula 07: a matriz é um array criado com uma **lista de linhas**;
`np.linalg.solve(A, b)` resolve $A\,x = b$; e `A @ x` refaz o lado esquerdo com a
resposta, e por isso tem de devolver `b`.

**✍️ Passo 1.** Monte `A` (uma linha por compra, uma coluna por fruta) e `b` (os totais) das compras da feira, resolva com `np.linalg.solve` e imprima também `A @ x`.

In [ ]:
# ✍️ passo 1

**Preveja:** os preços vão sair positivos? Algum vai passar de R$ 10?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Maçã a R\$ 1.50, banana a R\$ 2.00 e laranja a R\$ 3.00
o kg; `A @ x` refaz os totais 14; 13,5; 11,5. E os preços são de feira: a
conferência no **problema** também passa.

📖 [capítulo 9 · Uma receita em quatro passos](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/09-do-problema-ao-sistema/#uma-receita-em-quatro-passos)

</details>

## 2. No quadro: da tabela à matriz

Uma fábrica faz as peças P1, P2 e P3 em três máquinas e quer usar **todas** as horas
do mês.

| horas por peça | P1 | P2 | P3 | horas no mês |
|---|---|---|---|---|
| máquina M1 | 2 | 4 | 1 | 100 |
| máquina M2 | 3 | 2 | 4 | 140 |
| máquina M3 | 1 | 2 | 3 | 100 |

📖 [capítulo 9 · No quadro: da tabela à matriz](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/09-do-problema-ao-sistema/#no-quadro-da-tabela-a-matriz)

### 🧑‍🏫 No quadro — o que é linha e o que é coluna

Caderno de papel aberto. No quadro:

1. colunas = incógnitas: uma por peça;
2. linhas = informações: uma por máquina ("as horas desta máquina somam tanto");
3. o teste: **cada linha da matriz é uma equação?**
4. o erro clássico: a matriz **transposta**, que roda sem reclamar.

<details>
<summary><b>▶ O resumo do quadro</b></summary>

Máquina M1: $2x_0 + 4x_1 + 1x_2 = 100$. A tabela já é a matriz: linhas = máquinas,
colunas = peças. Transposta, as linhas seriam peças, e uma linha não seria mais uma
soma de horas de uma máquina.

</details>

**✍️ Passo 2.** Monte a matriz certa e resolva. Depois monte a **transposta** à mão (as linhas viram colunas), resolva de novo e calcule, com a matriz **certa**, quantas horas cada máquina usaria com essa resposta errada.

In [ ]:
# ✍️ passo 2

**Preveja:** a resposta da matriz transposta vai dar números absurdos (negativos, enormes) ou razoáveis?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Razoáveis — e é esse o perigo. A certa dá 10,00, 15,00, 20,00 peças, que usam
exatamente 100, 140 e 100 horas. A transposta dá 25,00, 15,00, 5,00 peças, que
usariam 115,00, 125,00, 70,00 horas: a máquina M1 passaria do limite. Só a conferência no
problema pega o erro.

📖 [capítulo 9 · No quadro: da tabela à matriz](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/09-do-problema-ao-sistema/#no-quadro-da-tabela-a-matriz)

</details>

## 3. Informações que não são somas

"B é o dobro de A" vira $x_B = 2x_A$, que arrumado fica $-2x_A + x_B = 0$: tudo o que
tem incógnita vai para a esquerda, e o lado direito fica zero.

📖 [capítulo 9 · Informações que não são somas](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/09-do-problema-ao-sistema/#informacoes-que-nao-sao-somas)

**✍️ Passo 3.** Três amigos pediram 3 sanduíches, 2 sucos e 4 salgados, por R$ 38. O sanduíche custa o dobro do suco, e o salgado, a metade do suco. Monte e resolva (colunas: sanduíche, suco, salgado).

In [ ]:
# ✍️ passo 3

**Preveja:** quantos números do lado direito, `b`, vão ser zero?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Dois: as duas comparações. Sanduíche R\$ 7.60, suco R\$ 3.80,
salgado R\$ 1.90. As linhas das comparações são `[1, -2, 0]` e
`[0, -0.5, 1]`.

📖 [capítulo 9 · Informações que não são somas](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/09-do-problema-ao-sistema/#informacoes-que-nao-sao-somas)

</details>

> ⚠️ **Armadilha.** "O salgado custa a metade do suco" é $x_{salg} = 0{,}5\,x_{suco}$, ou seja,
$-0{,}5\,x_{suco} + x_{salg} = 0$. Escrever $x_{suco} = 0{,}5\,x_{salg}$ inverte a
comparação: o sistema resolve sem reclamar, e o salgado fica mais caro que o suco.

### 🎯 Sua vez — A mistura no laboratório

Soluções A (20 % de ácido), B (35 %) e C (50 %); o volume de B é o dobro do de A.
Escreva `mistura(total, porcento)`, que devolve os litros de A, B e C para
preparar `total` litros com `porcento` % de ácido.

In [ ]:
def mistura(total, porcento):
    # sua solução aqui
    pass

In [ ]:
confere(mistura, [
    ((100.0, 32.0), [30.0, 60.0, 10.0]),
    ((50.0, 40.0), [8.33333333333333, 16.66666666666666, 25.00000000000001]),
])

<details>
<summary><b>💡 Dica</b></summary>

Três linhas: o volume `[1, 1, 1]`, o ácido `[0.20, 0.35, 0.50]` (com `porcento / 100 * total` à direita) e o estoque `[-2, 1, 0]` (com 0 à direita).

</details>

## 4. O que entra é o que sai

Em regime, em cada tanque, a massa que entra por minuto é igual à que sai, com massa
por minuto = vazão × concentração. Três tanques (vazões em L/min):

- **tanque 1**: entram 6 L/min a 10 g/L e 2 L/min do tanque 2; saem 8 L/min;
- **tanque 2**: entram 8 L/min do tanque 1 e 2 L/min de água pura; saem 10 L/min;
- **tanque 3**: entram 8 L/min do tanque 2 e 1 L/min a 20 g/L; saem 9 L/min.

📖 [capítulo 9 · O que entra é o que sai](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/09-do-problema-ao-sistema/#o-que-entra-e-o-que-sai)

**✍️ Passo 4.** Escreva, no papel, a equação de cada tanque (com $c_1$, $c_2$, $c_3$) e arrume. Depois monte e resolva.

In [ ]:
# ✍️ passo 4

**Preveja:** a concentração do tanque 3 vai ficar acima ou abaixo de 10 g/L?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Tanque 1: $6 \cdot 10 + 2c_2 = 8c_1$, ou $8c_1 - 2c_2 = 60$. As concentrações são
9,38, 7,50, 8,89 g/L. O tanque 3 fica abaixo de 10: a entrada de 20 g/L é pequena (1 L/min)
perto dos 8 L/min que chegam do tanque 2, já diluídos.

📖 [capítulo 9 · O que entra é o que sai](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/09-do-problema-ao-sistema/#o-que-entra-e-o-que-sai)

</details>

## 5. Circuitos: do desenho ao sistema

```
   ┌──── 10 Ω ────┬─────────────┬──── 15 Ω ────┐
   │              │             │              │
 (v1)      i1    5 Ω     i2    10 Ω     i3    (v3)
   │              │             │              │
   └──────────────┴──── 5 Ω ────┴──────────────┘
```

Em cada malha, a soma das quedas $R \cdot i$ é a tensão da fonte; num resistor
dividido por duas malhas, a corrente é a diferença das duas.

📖 [capítulo 9 · Circuitos: do desenho ao sistema](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/09-do-problema-ao-sistema/#circuitos-do-desenho-ao-sistema)

### 🎯 Sua vez — As correntes de malha

Escreva `malhas(v1, v3)`, que monta o sistema do circuito acima (a malha 2 não tem
fonte) e devolve `[i1, i2, i3]`. Escreva as três equações no papel antes.

In [ ]:
def malhas(v1, v3):
    # sua solução aqui
    pass

In [ ]:
confere(malhas, [
    ((10.0, 5.0), [0.7906976744186046, 0.37209302325581395, 0.3488372093023256]),
    ((12.0, 0.0), [0.8930232558139536, 0.2790697674418605, 0.1116279069767442]),
])

<details>
<summary><b>💡 Dica</b></summary>

Malha 1: $10\,i_1 + 5\,(i_1 - i_2) = v_1$, ou seja, a linha `[15, -5, 0]`. Na diagonal, a soma dos resistores da malha; fora dela, menos o resistor dividido.

</details>

## 6. Forças em equilíbrio

Parado, o ponto onde os cabos se encontram tem forças somando zero na horizontal e na
vertical. Uma tração $T$ num cabo com ângulo $\theta$ (em relação à horizontal) tem as
partes $T\cos\theta$ e $T\sin\theta$. Para os ângulos em radianos: `graus * np.pi / 180`.

📖 [capítulo 9 · Forças em equilíbrio](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/09-do-problema-ao-sistema/#forcas-em-equilibrio)

**✍️ Passo 5.** Uma placa de 300 N é pendurada por dois cabos: o da esquerda a 40° da horizontal, o da direita a 60°. Monte as equações horizontal e vertical e resolva para $T_1$ e $T_2$.

In [ ]:
# ✍️ passo 5

**Preveja:** qual cabo segura mais: o mais inclinado (60°) ou o mais deitado (40°)?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

$T_1 = 152.3$ N (40°) e $T_2 = 233.4$ N (60°): o cabo mais em pé
segura mais, porque a parte vertical dele é maior. A linha horizontal é
`[-cos 40°, cos 60°]` com 0 à direita; a vertical, `[sin 40°, sin 60°]` com 300.

📖 [capítulo 9 · Forças em equilíbrio](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/09-do-problema-ao-sistema/#forcas-em-equilibrio)

</details>

## 7. Quando a informação não basta

📖 [capítulo 9 · Quando a informação não basta](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/09-do-problema-ao-sistema/#quando-a-informacao-nao-basta)

**✍️ Passo 6.** Troque a terceira compra da feira por 3 kg de maçã, 4 de banana e 5 de laranja, por R$ 27,50, resolva e confira com `A @ x`.

In [ ]:
# ✍️ passo 6

**Preveja:** o `solve` vai dar os mesmos preços de antes, outros preços, ou uma mensagem de erro?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Outros preços (2,90, 2,20, 2,00), sem mensagem de erro — e o `A @ x` confere! A
compra nova é a **soma** das duas primeiras: não traz informação nova, e há
infinitos preços possíveis. Com R\$ 27,60 (as informações se contradizem), os
preços vão a $10^{14}$.

📖 [capítulo 9 · Quando a informação não basta](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/09-do-problema-ao-sistema/#quando-a-informacao-nao-basta)

</details>

## 8. Mesmo método, outra área

**Aeroespacial.** A velocidade de um foguete foi medida em 5, 8 e 12 s: 106,8,
177,2 e 279,2 m/s. A parábola $v(t) = a\,t^2 + b\,t + c$ passa pelas três medidas.
As incógnitas agora são os **coeficientes**.

📖 [capítulo 9 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/09-do-problema-ao-sistema/#mesmo-metodo-outra-area)

**✍️ Passo 7.** Monte a matriz (uma linha por medida: $t^2$, $t$, 1), ache $a$, $b$ e $c$ e estime a velocidade em 10 s.

In [ ]:
# ✍️ passo 7

**Preveja:** a estimativa em 10 s vai ficar entre 177 e 279 m/s?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Sim: 227.0 m/s. É a mesma montagem das compras, com outro significado para as
linhas; essa matriz de potências volta na interpolação (matriz de Vandermonde).

📖 [capítulo 9 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade4-sistemas-lineares/09-do-problema-ao-sistema/#mesmo-metodo-outra-area)

</details>

## 🎯 Prática

A prática desta aula é o problema, logo abaixo: montar o sistema do quarteirão.

## 🧩 Resolvendo o problema

> *"**Quantos carros passam nas outras três ruas?**"* — a prefeitura.

Em cada cruzamento, o que chega é o que sai. No cruzamento A chegam 400 carros de
fora e os $x_4$ da rua de D para A; saem 150 para fora e os $x_1$ para B:
$400 + x_4 = x_1 + 150$, ou $x_1 - x_4 = 250$.

**✍️ Passo 8.** Escreva as equações dos **quatro** cruzamentos, monte a matriz 4 × 4 só com elas e tente resolver.

In [ ]:
# ✍️ passo 8

**Preveja:** quatro cruzamentos, quatro incógnitas: vai dar certo?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Não: o `solve` para com `LinAlgError: Singular matrix`. Somando
as quatro equações, tudo se cancela: cada carro que entra no anel por um
cruzamento sai por outro. A quarta equação **repete** as outras três. Falta
informação — e é para isso que serve o sensor.

</details>

### 🎯 Sua vez — As ruas sem sensor

Escreva `fluxos(x4)`, que usa as equações dos cruzamentos **A, B e C** e a do
**sensor** ($x_4$ = valor medido) e devolve `[x1, x2, x3, x4]`. A equação do
cruzamento D fica para conferir.

In [ ]:
def fluxos(x4):
    # sua solução aqui
    pass

In [ ]:
confere(fluxos, [
    ((180.0,), [430.0, 480.0, 330.0, 180.0]),
    ((260.0,), [510.0, 560.0, 410.0, 260.0]),
])

<details>
<summary><b>💡 Dica</b></summary>

A linha do sensor é `[0, 0, 0, 1]`, com `x4` à direita. Cruzamento B: $x_1 + 300 = x_2 + 250$, ou seja, `[-1, 1, 0, 0]` com 50 à direita.

</details>

In [ ]:
resposta = fluxos(180.0)
if resposta is not None:
    print("carros por hora nas ruas A->B, B->C, C->D, D->A:", resposta)
    print("confere o cruzamento D:", resposta[2] + 100, "=", resposta[3] + 250)

<details>
<summary><b>▶ O que os números dizem</b></summary>

430 carros por hora de A para B, 480 de B para C e 330
de C para D. A rua de B para C é a mais carregada: é lá que o semáforo novo faz mais
diferença. O cruzamento D confere (330 + 100 = 180 + 250).

A lição que fica: **quatro equações não são quatro informações**. Numa rede fechada,
a conservação num dos nós é consequência das outras. Toda rede (de ruas, de canos, de
dados) precisa de pelo menos uma medida no anel.

</details>

## 📋 A lista

Abra a [Lista 09](https://lacouth.github.io/metodos_telecom-site/listas/lista09/). O **Exercício 01** é à mão (✏️): montar o sistema dos
ingressos de um festival e conferir uma resposta. Comece por ele, no papel.

**a)** No ponto de venda 1 foram 20 VIP, 35 regulares e 40 estudantes, por R\$ 6.300.
Qual é a primeira linha de $A$ e o primeiro número de $b$?

<details>
<summary><b>▶ Resposta</b></summary>

A linha `[20, 35, 40]` (uma coluna por tipo de ingresso) e 6300.

</details>

Termine o exercício e siga para o **Exercício 02**, a conta da pizzaria.

## 🚪 Antes de sair

**1.** Como saber se uma tabela do enunciado já é a matriz, ou a transposta dela?

<details>
<summary><b>▶ Resposta da 1</b></summary>

Pergunte se cada **linha** é uma equação: uma soma (das incógnitas vezes os números da linha) que dá um número de `b`. Se for, é a matriz; se as equações estão nas colunas, é a transposta.

</details>

**2.** Por que três compras podem não bastar para descobrir três preços?

<details>
<summary><b>▶ Resposta da 2</b></summary>

Porque uma delas pode ser combinação das outras (a soma, por exemplo). Aí são três equações, mas só duas informações, e há infinitas soluções.

</details>

**3.** O `A @ x` conferiu. A resposta está certa?

<details>
<summary><b>▶ Resposta da 3</b></summary>

Está certa **para o sistema**. Ainda falta conferir no problema: faz sentido (sinal, tamanho, unidade)? A montagem pode estar errada, ou o sistema pode ter infinitas soluções.

</details>

## 🏠 Para casa

- Termine a [Lista 09](https://lacouth.github.io/metodos_telecom-site/listas/lista09/).
- A Unidade 4 termina aqui. A Unidade 5 começa com uma pergunta: e o valor **entre**
  os pontos de uma tabela?